In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [58]:
!pip install -q \
    transformers==4.40.2 \
    accelerate==0.30.1 \
    sentencepiece \
    bitsandbytes \
    pandas \
    numpy \
    tqdm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [59]:
import os
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [60]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

patient_notes = pd.read_csv(f"{NBME_PATH}/patient_notes.csv")
train = pd.read_csv(f"{NBME_PATH}/train.csv")

print("Total notes:", len(patient_notes))
print("Total annotations:", len(train))


Total notes: 42146
Total annotations: 14300


In [61]:
annotated_pn_nums = set(train["pn_num"].unique())

nbme_annotated = patient_notes[
    patient_notes["pn_num"].isin(annotated_pn_nums)
][["pn_num", "pn_history"]].reset_index(drop=True)

print("Annotated notes only:", len(nbme_annotated))


Annotated notes only: 1000


In [62]:
MAX_CHARS = 1500

def truncate_note(text, max_chars=MAX_CHARS):
    if not isinstance(text, str):
        return ""
    return text[:max_chars]

BASE_PROMPT = """### SYSTEM
You are a clinical information extraction system.
You must follow the rules exactly.

Rules:
- Only extract patient-reported symptoms or clinician-observed findings.
- Use ONLY the clinical note provided by the user.
- Do NOT include diagnoses, medications, procedures, labs, demographics, family history, ROS, PMH, or social history.
- Do NOT include negated symptoms.
- Do NOT infer or assume symptoms.
- Output must be either:
  (a) one or more symptoms, each on its own line starting with '-'
  (b) exactly the word 'none'

### USER
Clinical note:
{note}

### ASSISTANT
"""

def format_prompt(note_text):
    return BASE_PROMPT.format(note=truncate_note(note_text))


In [63]:
def parse_symptoms(output_text):
    if not isinstance(output_text, str):
        return []

    if "### ASSISTANT" in output_text:
        output_text = output_text.split("### ASSISTANT", 1)[1]

    output_text = output_text.strip().lower()

    if output_text == "none":
        return []

    symptoms = []
    for line in output_text.splitlines():
        line = line.strip()
        if line.startswith("###"):
            break
        if line.startswith("-"):
            s = line[1:].strip()
            if s:
                symptoms.append(s)

    seen = set()
    unique = []
    for s in symptoms:
        if s not in seen:
            seen.add(s)
            unique.append(s)

    return unique


In [64]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )
    (norm): MistralRMSNorm(

In [65]:
BATCH_SIZE = 4          # Safe on Kaggle T4 for Mistral
MAX_NEW_TOKENS = 64
CHECKPOINT_EVERY = 100

OUTPUT_DIR = "/kaggle/working/nbme_mistral_annotated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_DIR}/mistral_predictions_annotated.csv"


In [66]:
processed_ids = set()

if os.path.exists(OUTPUT_FILE):
    prev = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(prev["pn_num"].tolist())
    print("Resuming from", len(processed_ids), "notes")
else:
    print("Starting fresh inference")


Starting fresh inference


In [ ]:
results = []
total_notes = len(nbme_annotated)

for start_idx in tqdm(range(0, total_notes, BATCH_SIZE)):
    batch = nbme_annotated.iloc[start_idx:start_idx + BATCH_SIZE]
    batch = batch[~batch["pn_num"].isin(processed_ids)]

    if batch.empty:
        continue

    prompts = [format_prompt(t) for t in batch["pn_history"].tolist()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    for pn, out in zip(batch["pn_num"], decoded):
        results.append({
            "pn_num": pn,
            "model": "mistral-7b-instruct",
            "predicted_symptoms": parse_symptoms(out)
        })

    if len(results) >= CHECKPOINT_EVERY:
        df = pd.DataFrame(results)
        if os.path.exists(OUTPUT_FILE):
            df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)
        else:
            df.to_csv(OUTPUT_FILE, index=False)

        processed_ids.update(df["pn_num"].tolist())
        results = []
        torch.cuda.empty_cache()


  0%|          | 0/250 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting

In [ ]:
if results:
    df = pd.DataFrame(results)
    if os.path.exists(OUTPUT_FILE):
        df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)
    else:
        df.to_csv(OUTPUT_FILE, index=False)

print("Mistral annotated NBME inference complete.")


In [ ]:
df = pd.read_csv(OUTPUT_FILE)
print("Total predictions:", len(df))
df.head()


In [9]:
import pandas as pd
import os
import json

In [10]:
PRED_PATH = "/kaggle/input/mistral-nbme/mistral_predictions_annotated.csv"

assert os.path.exists(PRED_PATH), "Prediction file not found"

preds = pd.read_csv(PRED_PATH)

print("Predictions:", preds.shape)
preds.head()


Predictions: (1000, 3)


,pn_num,model,predicted_symptoms
0,16,mistral-7b-instruct,"['palpitations', 'chest pressure', 'passing ou..."
1,41,mistral-7b-instruct,"['heart pounding', 'shortness of breath', 'pre..."
2,46,mistral-7b-instruct,"['palpitations', 'light headedness', 'shortnes..."
3,82,mistral-7b-instruct,"['heart racing', 'heart pounding', 'light head..."
4,100,mistral-7b-instruct,"['intermittent tachycardia', 'pounding heart b..."


In [6]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict


In [12]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

train = pd.read_csv(f"{NBME_PATH}/train.csv")
features = pd.read_csv(f"{NBME_PATH}/features.csv")

print("Train:", train.shape)
print("Features:", features.shape)


Train: (14300, 6)
Features: (143, 3)


In [13]:
feature_map = dict(zip(features["feature_num"], features["feature_text"]))
train["feature_text"] = train["feature_num"].map(feature_map)

gt_df = (
    train.groupby("pn_num")["feature_text"]
    .apply(list)
    .reset_index()
)

gt_df.head()


,pn_num,feature_text
0,16,[Family-history-of-MI-OR-Family-history-of-myo...
1,41,[Family-history-of-MI-OR-Family-history-of-myo...
2,46,[Family-history-of-MI-OR-Family-history-of-myo...
3,82,[Family-history-of-MI-OR-Family-history-of-myo...
4,100,[Family-history-of-MI-OR-Family-history-of-myo...


In [14]:
eval_df = preds.merge(gt_df, on="pn_num", how="inner")

eval_df = eval_df.rename(columns={
    "predicted_symptoms": "predicted",
    "feature_text": "gold"
})

print("Evaluation notes:", len(eval_df))
eval_df.head()


Evaluation notes: 1000


,pn_num,model,predicted,gold
0,16,mistral-7b-instruct,"['palpitations', 'chest pressure', 'passing ou...",[Family-history-of-MI-OR-Family-history-of-myo...
1,41,mistral-7b-instruct,"['heart pounding', 'shortness of breath', 'pre...",[Family-history-of-MI-OR-Family-history-of-myo...
2,46,mistral-7b-instruct,"['palpitations', 'light headedness', 'shortnes...",[Family-history-of-MI-OR-Family-history-of-myo...
3,82,mistral-7b-instruct,"['heart racing', 'heart pounding', 'light head...",[Family-history-of-MI-OR-Family-history-of-myo...
4,100,mistral-7b-instruct,"['intermittent tachycardia', 'pounding heart b...",[Family-history-of-MI-OR-Family-history-of-myo...


In [15]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_list(items):
    return [normalize_text(x) for x in items if isinstance(x, str)]


In [16]:
def is_match(pred, gold):
    return pred == gold or pred in gold or gold in pred


In [17]:
records = []

for _, row in eval_df.iterrows():
    pred_list = normalize_list(eval(row["predicted"]))
    gold_list = normalize_list(row["gold"])

    matched_gold = set()
    tp = 0

    for p in pred_list:
        for g in gold_list:
            if g not in matched_gold and is_match(p, g):
                matched_gold.add(g)
                tp += 1
                break

    fp = max(len(pred_list) - tp, 0)
    fn = max(len(gold_list) - tp, 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    records.append({
        "pn_num": row["pn_num"],
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

per_note_df = pd.DataFrame(records)
per_note_df.head()


,pn_num,tp,fp,fn,precision,recall,f1
0,16,1,5,12,0.166667,0.076923,0.105263
1,41,2,1,11,0.666667,0.153846,0.250000
2,46,1,4,12,0.200000,0.076923,0.111111
3,82,2,2,11,0.500000,0.153846,0.235294
4,100,1,3,12,0.250000,0.076923,0.117647


In [18]:
mean_precision = per_note_df["precision"].mean()
mean_recall = per_note_df["recall"].mean()
mean_f1 = per_note_df["f1"].mean()

print(f"Precision: {mean_precision:.4f}")
print(f"Recall:    {mean_recall:.4f}")
print(f"F1-score:  {mean_f1:.4f}")


Precision: 0.3243
Recall:    0.1196
F1-score:  0.1695


In [19]:
def bootstrap_ci(values, n_bootstrap=1000, alpha=0.05):
    rng = np.random.default_rng(42)
    samples = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        samples.append(sample.mean())
    lower = np.percentile(samples, 100 * (alpha / 2))
    upper = np.percentile(samples, 100 * (1 - alpha / 2))
    return lower, upper


prec_ci = bootstrap_ci(per_note_df["precision"].values)
rec_ci = bootstrap_ci(per_note_df["recall"].values)
f1_ci = bootstrap_ci(per_note_df["f1"].values)

print("Precision CI:", prec_ci)
print("Recall CI:", rec_ci)
print("F1 CI:", f1_ci)


Precision CI: (np.float64(0.31085750471750473), np.float64(0.33935269043456545))
Recall CI: (np.float64(0.1148229264077426), np.float64(0.12448678151709401))
F1 CI: (np.float64(0.16294369032423164), np.float64(0.17666421376029198))


In [20]:
OUT_DIR = "/kaggle/working/nbme_mistral_eval"
os.makedirs(OUT_DIR, exist_ok=True)

per_note_df.to_csv(f"{OUT_DIR}/mistral_per_note_scores.csv", index=False)

metrics = {
    "dataset": "nbme",
    "model": "mistral-7b-instruct",
    "n_notes": len(per_note_df),
    "precision": mean_precision,
    "recall": mean_recall,
    "f1": mean_f1,
    "precision_ci": prec_ci,
    "recall_ci": rec_ci,
    "f1_ci": f1_ci
}

with open(f"{OUT_DIR}/mistral_nbme_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Mistral NBME evaluation saved.")


Mistral NBME evaluation saved.
